# Approach Ball rollout

This tutorial runs the H44 state-probe evaluation and inspects its machine-readable metrics. Provide a locally trained or converted Approach model/probe pair, generate the evaluation data locally, then edit the **Settings** cell below. No shell environment variables are required.

In [ ]:
from pathlib import Path

import torch

from sg_jepa.evaluation import run_frozen_approach_evaluation

# Settings: edit these paths after downloading the model artifacts and generating the data.
MODEL_KIND = "native"  # Choose "native" or "dino".
METHOD = "SG-JEPA (GRU)"
CONFIG = Path("configs/train/approach_sg_jepa_gru.yaml")
CHECKPOINT = Path("artifacts/approach/sg-jepa-gru-epoch20.safetensors")
PROBE = Path("artifacts/approach/sg-jepa-gru-state-probe.safetensors")
PROBE_CONFIG = Path("artifacts/approach/sg-jepa-gru-state-probe.json")
DATA = Path("datasets/approach-ball-evaluation")
DINOV2_ROOT = None  # Set to a Path only when MODEL_KIND == "dino".
OUTPUT = Path("outputs/approach-rollout")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_EPISODES = 8

required = {
    "model config": CONFIG,
    "model checkpoint": CHECKPOINT,
    "probe checkpoint": PROBE,
    "probe config": PROBE_CONFIG,
    "evaluation dataset": DATA,
}
missing = [name for name, path in required.items() if not path.exists()]
if MODEL_KIND == "dino" and (DINOV2_ROOT is None or not DINOV2_ROOT.exists()):
    missing.append("DINOv2 checkout and weights")
READY = not missing
print(
    "Ready to evaluate."
    if READY
    else f"Edit the Settings cell after provisioning: {', '.join(missing)}"
)

In [ ]:
report = None
if READY:
    RUN_OUTPUT = OUTPUT
    run_number = 2
    while RUN_OUTPUT.exists():
        RUN_OUTPUT = OUTPUT.with_name(f"{OUTPUT.name}-{run_number}")
        run_number += 1
    report = run_frozen_approach_evaluation(
        method=METHOD,
        model_kind=MODEL_KIND,
        config_path=CONFIG,
        checkpoint_path=CHECKPOINT,
        probe_path=PROBE,
        probe_config_path=PROBE_CONFIG,
        dataset_path=DATA,
        output_dir=RUN_OUTPUT,
        device=DEVICE,
        max_episodes=MAX_EPISODES,
        horizon=44,
        dinov2_root=DINOV2_ROOT,
    )
    print(f"Wrote {RUN_OUTPUT / 'evaluation.json'}")

In [ ]:
if report is not None:
    import csv

    summary = {
        key: report[key] for key in ("method", "horizon", "evaluated_episode_count", "status")
    }
    print(summary)
    with Path(report["outputs"]["horizon_metrics"]).open() as handle:
        rows = list(csv.DictReader(handle))
    print("First three aggregate horizon records:")
    for row in rows[:3]:
        print(row)